In [ ]:
# 9회차 — 화염 소재 라이브러리 교체 (백열 코어 보존 + 백색 화염 영상 7개 추가)
# 변수는 화염 소재 하나. 나머지는 5회차와 동일 (fire 단일 클래스, imgsz 640, 배치 16)
# 사전 등록: docs/PREREGISTER_R9.md
print('round9 notebook v1')
!nvidia-smi -L
!pip -q install ultralytics==8.3.* kagglehub


In [ ]:
# [1] 업로드 — 아래 7개를 한 번에 선택
#   kitchen-fire-poc.zip
#   assets_1_bases.zip · assets_3_negsrc.zip · assets_4_weights.zip
#   flamelib2_a.zip · flamelib2_b.zip
#   eval_F_fire.zip · eval_F_nofire.zip
import zipfile, os, glob
from google.colab import files
up = files.upload()
os.makedirs('/content/work', exist_ok=True)
for n in up:
    zipfile.ZipFile(n).extractall('/content/work')
os.chdir('/content/work')
for d in ('bases', 'flamelib2', 'negsrc', 'eval_neg', 'weights'):
    p = f'assets/{d}'
    print(f'{d:12s}', len(glob.glob(p + '/*')) if os.path.isdir(p) else '없음')
print('평가군 F 화염   ', len(glob.glob('eval_F/F_fire/*.jpg')))
print('평가군 F 화염없음', len(glob.glob('eval_F/F_nofire/*.jpg')))
assert len(glob.glob('assets/flamelib2/*.webp')) == 723, '소재 723종이 아닙니다 — zip 두 개를 모두 올렸는지 확인'


In [ ]:
# [2] D-Fire
import kagglehub
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DFIRE)


In [ ]:
# [3] 평가셋 A·C + 학습용 배경 (5회차와 동일 절차)
!python scripts/dfire_eval_set.py --dfire "$DFIRE" --out eval


In [ ]:
# [4] 합성 — 5회차 구성 그대로, 소재만 flamelib2 (723종)
!python scripts/synthesize.py --assets assets --flamelib flamelib2 --out ds \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600 --haze-prob 0.5
!cat ds/data.yaml


In [ ]:
# [5] 합성 결과 눈으로 확인 — 백열 코어가 살아 있는지, 라벨이 맞는지
import glob, cv2, numpy as np, random
from google.colab.patches import cv2_imshow
random.seed(9)
ps = random.sample(sorted(glob.glob('ds/images/train/*.jpg')), 8)
tiles = []
for p in ps:
    im = cv2.imread(p); h, w = im.shape[:2]
    lp = p.replace('/images/', '/labels/').replace('.jpg', '.txt')
    for ln in open(lp):
        _, x, y, bw, bh = map(float, ln.split())
        x0, y0 = int((x - bw / 2) * w), int((y - bh / 2) * h)
        x1, y1 = int((x + bw / 2) * w), int((y + bh / 2) * h)
        cv2.rectangle(im, (x0, y0), (x1, y1), (0, 255, 0), 2)
    tiles.append(cv2.resize(im, (400, 225)))
cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, 8, 4)]))


In [ ]:
# [6] 학습 — imgsz 640, 배치 16 (5회차와 동일, 약 40분)
!yolo detect train model=yolov8s.pt data=ds/data.yaml epochs=60 imgsz=640 \
    batch=16 project=/content/runs name=r9 exist_ok=False


In [ ]:
# [7] 채점 A·B·C — 5회차 기준선과 나란히
import glob, os
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('=' * 64); print('[5회차 기준선]'); print('=' * 64)
!python scripts/eval_gate.py --weights assets/weights/round5_best.pt \
    --eval-dir eval --cctv assets/eval_neg --conf 0.10
print('\n' + '=' * 64); print('[9회차 · 소재 723종]'); print('=' * 64)
!python scripts/eval_gate.py --weights "$best" --eval-dir eval --cctv assets/eval_neg --conf 0.10


In [ ]:
# [8] 주 지표 — 평가군 F (주방 화재). 두 가중치를 같은 기준으로 채점
import glob, os, math, numpy as np, cv2
from ultralytics import YOLO

FIRE, CONF = 0, 0.10
P = sorted(glob.glob('eval_F/F_fire/*.jpg'))
N = sorted(glob.glob('eval_F/F_nofire/*.jpg'))
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)

def scan(m, paths, batch=32):
    out = []
    for i in range(0, len(paths), batch):
        for r in m.predict(paths[i:i+batch], conf=0.03, verbose=False):
            f = 0.0
            if len(r.boxes):
                cl = r.boxes.cls.cpu().numpy().astype(int)
                cf = r.boxes.conf.cpu().numpy()
                if (cl == FIRE).any():
                    f = float(cf[cl == FIRE].max())
            out.append(f)
    return np.array(out)

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (c-h)*100, (c+h)*100

TAGS = sorted({os.path.basename(q).split('_')[0] for q in P + N})
res = {}
for tag_w, w in (('5회차', 'assets/weights/round5_best.pt'), ('9회차', best)):
    m = YOLO(w)
    zP, zN = scan(m, P), scan(m, N)
    rows = []
    for t in TAGS:
        ip = [i for i, q in enumerate(P) if os.path.basename(q).startswith(t)]
        inn = [i for i, q in enumerate(N) if os.path.basename(q).startswith(t)]
        rec = float((zP[ip] >= CONF).mean()) if ip else None
        # 사전 등록 개정: 화염없음 10장 이상인 출처만 macro 오탐률에 포함
        fpr = float((zN[inn] >= CONF).mean()) if len(inn) >= 10 else None
        rows.append((t, len(ip), rec, len(inn), fpr))
    mrec = float(np.mean([r for _, _, r, _, _ in rows if r is not None]))
    mfpr = float(np.mean([f for _, _, _, _, f in rows if f is not None]))
    res[tag_w] = (mrec, mfpr, zP, zN, rows)

print('=' * 66)
print(f"{'':10s}{'F 인식률(macro)':>18s}{'F 오탐률(macro)':>18s}{'판별비 F':>12s}")
for k, (mrec, mfpr, zP, zN, rows) in res.items():
    print(f'{k:10s}{mrec*100:17.1f}%{mfpr*100:17.1f}%{mrec/mfpr if mfpr else float("inf"):12.2f}')
d = (res['9회차'][0] - res['5회차'][0]) * 100
print(f'\n인식률 변화 {d:+.1f}%p  →  ' +
      ('채택' if d >= 8 else '부분' if d >= 3 else '기각') + ' (사전 등록 기준)')
dfp = (res['9회차'][1] - res['5회차'][1]) * 100
print(f'오탐률 변화 {dfp:+.1f}%p  →  ' + ('기각 조건 위반' if dfp > 5 else '기준 이내'))

for k in res:
    mrec, mfpr, zP, zN, rows = res[k]
    kP = int((zP >= CONF).sum()); kN = int((zN >= CONF).sum())
    lo, hi = wilson(kP, len(P))
    print(f'\n[{k}] micro 인식률 {kP}/{len(P)} = {kP/len(P)*100:.1f}% (95%CI {lo:.1f}~{hi:.1f}) · '
          f'micro 오탐률 {kN}/{len(N)} = {kN/len(N)*100:.1f}%')
    for t, np_, rec, nn_, fpr in rows:
        a = f'{round(rec*np_)}/{np_} = {rec*100:5.1f}%' if rec is not None else '-'
        b = f'{round(fpr*nn_)}/{nn_} = {fpr*100:5.1f}%' if fpr is not None else f'(n={nn_}, 제외)'
        print(f'   {t:10s} 화염 {a:>16s}   화염없음 {b:>18s}')


In [ ]:
# [9] 화염 면적비 구간별 — 작은 화염과 큰 백열 화염을 각각 살렸는지
import numpy as np, cv2
def warm(p):
    im = cv2.imread(p); hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[...,0].astype(np.float32), hsv[...,1]/255., hsv[...,2]/255.
    return float((((h < 28) | (h > 170)) & (s > 0.35) & (v > 0.62)).mean())
ff = np.array([warm(p) for p in P])
print(f"{'면적비':>12s}{'5회차':>12s}{'9회차':>12s}")
for lo_, hi_ in ((0, .02), (.02, .05), (.05, .15), (.15, 1.0)):
    idx = np.where((ff >= lo_) & (ff < hi_))[0]
    if not len(idx): continue
    a = (res['5회차'][2][idx] >= CONF).mean() * 100
    b = (res['9회차'][2][idx] >= CONF).mean() * 100
    print(f'{lo_*100:5.0f}~{hi_*100:4.0f}%{a:11.1f}%{b:11.1f}%   (n={len(idx)})')


In [ ]:
# [10] 새로 잡은 것 / 새로 놓친 것 — 수치가 아니라 사진으로 확인
# 8회차와 F 첫 채점에서 두 번 다 이 단계가 결론을 바꿨음
import cv2, numpy as np
from google.colab.patches import cv2_imshow
z5, z9 = res['5회차'][2], res['9회차'][2]
gained = [P[i] for i in np.where((z9 >= CONF) & (z5 < CONF))[0]]
lost   = [P[i] for i in np.where((z9 < CONF) & (z5 >= CONF))[0]]
print(f'9회차가 새로 잡은 것 {len(gained)}장 · 새로 놓친 것 {len(lost)}장')
m9 = YOLO(best)
def sheet(paths, title, n=12):
    if not paths: print(f'\n■ {title} 없음'); return
    tiles = []
    for p in paths[:n]:
        im = cv2.imread(p)
        for r in m9.predict(p, conf=CONF, verbose=False):
            for b, c in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()):
                cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
                cv2.putText(im, f'{c:.2f}', (b[0], max(14, b[1]-4)),
                            cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 255, 0), 2)
        tiles.append(cv2.resize(im, (400, 225)))
    while len(tiles) % 4: tiles.append(np.zeros((225, 400, 3), np.uint8))
    print(f'\n■ {title} ({len(paths)}장 중 앞 {min(n,len(paths))}장)')
    cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, len(tiles), 4)]))
sheet(gained, '9회차가 새로 잡은 화염')
sheet(lost, '9회차가 새로 놓친 화염')


In [ ]:
# [11] 가중치 내려받기
import glob, os
from google.colab import files
files.download(max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime))
